[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.06.notebook-title|Demonstration -->
# Week 6 Demonstration: DataViz I: Visual encoding and chart choice
<!-- /gen -->

*Each example in this notebook relates the [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate)'s first draft of a chart (in matplotlib and seaborn) with the revision for which you are responsible.*

**How to work through this notebook.** Each example combines: (a) The *Associate's first draft* (from a single-line prompt); (b) the *revision* you direct, with the exact prompt that produced it.

Work with your Associate. Try each revision prompt yourself. Your Associate's charts will differ from the ones this notebook draws. Note what your Associate does by default, and what you have to correct. If your Associate is not set up, the [workstation guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/codespaces-start.md) explains how. The [AI Lens](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-lens) callouts give more prompts to try.

**The client:** Aldergate Books, a five-branch bookshop chain preparing its eighteen-month trading review.

The cell below builds Aldergate's eighteen months of till data in memory with the module seed, plus the marketing agency's weekly campaign tracker. Run it first.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(5813)  # module seed: Do not change

# Aldergate Books: Five branches, eighteen months of till data.
BRANCHES = ["Egham", "Staines", "Windsor", "Kingston", "Guildford"]
n = 4000
dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 546, n), unit="D")
branch = rng.choice(BRANCHES, size=n, p=[0.18, 0.14, 0.28, 0.16, 0.24])
days = (dates - dates.min()).days.to_numpy()
online_share = 0.18 + 0.20 * days / days.max()          # online grows over the period
channel = np.where(rng.random(n) < online_share, "Online", "In store")
order_value = np.round(rng.gamma(2.2, 6.4, n) + np.where(branch == "Windsor", 3.0, 0.0), 2)
satisfaction = np.clip(rng.normal(np.where(branch == "Kingston", 6.6, 7.9),
                                  np.where(branch == "Kingston", 1.6, 1.0)), 0, 10).round(1)

orders = pd.DataFrame({"date": dates, "branch": branch, "channel": channel,
                       "order_value": order_value, "satisfaction": satisfaction}
                      ).sort_values("date", ignore_index=True)

# The agency's weekly campaign tracker (a separate feed)
rng_w = np.random.default_rng(5813 + 1)  # derived seed for an independent dataset
spend = rng_w.uniform(300, 1500, 78).round()
weekly = pd.DataFrame({"week": pd.date_range("2025-01-05", periods=78, freq="W"),
                       "marketing_spend": spend,
                       "revenue": (9_000 + 4.2 * spend + rng_w.normal(0, 1_100, 78)).round()})

print(f"Created (in memory, seed 5813): orders ({len(orders)} rows, "
      f"{orders['date'].min():%b %Y}–{orders['date'].max():%b %Y}) · weekly ({len(weekly)} weeks)")

The cell below shows a <!-- plain-language: everyday first look -->first look at the till data. Each row is one order, with its date, branch, channel, order value, and a satisfaction score out of ten.

In [ ]:
orders.head()

## The basics: `Figure` and `Axes`

In matplotlib, the `Figure` is the **page** and an `Axes` is one plotting **area** on that page. A `Figure` can contain several `Axes`. Most of what you do to a chart, you do through methods on the `Axes`: `ax.plot(...)`, `ax.set_title(...)`, `ax.set_xlabel(...)`. The [matplotlib quick start](https://matplotlib.org/stable/users/explain/quick_start.html) shows you how to do this in more detail.

There are **two styles**. The *state-machine* style (`plt.plot(...)`, `plt.title(...)`) acts on whatever the current chart is. Once a page contains more than one chart, this style cannot tell which chart you mean. The *object-oriented* style (`fig, ax = plt.subplots()`, then methods on `ax`) always names the chart *it acts on*. **Use this second style**. Ask your Associate for it in these words: "Use the object-oriented interface, fig and ax".

The cell below draws one line in the object-oriented style.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))  # Ask your Associate why we assign values to TWO variables with a single action

# Or, alternative formatting:
# fig, ax = plt.subplots(
#     figsize=(7, 3.5))

ax.plot([1, 2, 3, 4], [10, 12, 9, 14], color="#0072B2", marker="o")  # Try different marker values

ax.set_title("Every chart this week starts like this", loc="left")
ax.set_xlabel("x: whatever runs along the bottom")
ax.set_ylabel("y: whatever runs up the side")

plt.show()

Additional terminology you need to direct your Associate:

- **Spines:** The four lines framing the plot. The prompts below turn the top and right ones off
- **Ticks** and **tick labels:** The scale markings
- **Legend:** The floating box that names each colour or line
- **Annotations:** Text placed on the chart itself

*(This notebook takes its accent colours from the **Okabe-Ito** colour-blind-safe palette, such as `#0072B2` (blue) and `#D55E00` (vermillion). Everything else is grey. The [design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md) provides further detail.)*

## Pair 1: Comparison → bar chart

**The business question:** Which branch earns Aldergate the most?

This is a *comparison* task. The reader needs every value to start from the same line, so use bars.

> **The prompt that produced the draft:** *"Plot revenue by branch from the orders `DataFrame`."*

In [ ]:
# Associate's first draft
branch_rev = orders.groupby("branch")["order_value"].sum()

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(branch_rev.index, branch_rev, color=plt.cm.tab10.colors[:5])
ax.legend(bars, branch_rev.index, title="branch")
ax.set_title("Revenue by Branch")
ax.grid(True)
plt.show()

The draft shows these habits:

- A different colour for each bar, and the colours mean nothing
- A legend repeating what the x-axis already shows
- Alphabetical order
- No units
- A full grid
- A title that only names the columns

The revision sorts horizontal bars by value and uses [`ax.bar_label`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.bar_label.html) to print each value on its bar.

> **The prompt that fixed it:** *"Revise: Horizontal bars sorted by value, one neutral grey with the leader in Okabe-Ito blue #0072B2, no legend. Label each bar with its value in £k, x-axis labelled 'Revenue (£k)', top and right spines off, and a title that states the finding. Add a small source line."*

Here `colours` comes from a conditional list comprehension that picks blue for the top branch and grey for the rest (optional background: [Week 2b](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02b/Introduction.ipynb)).

In [ ]:
# Human-directed revision
branch_rev = orders.groupby("branch")["order_value"].sum().div(1000).sort_values()
colours = ["#0072B2" if b == branch_rev.idxmax() else "#949494" for b in branch_rev.index]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(branch_rev.index, branch_rev, color=colours)
ax.bar_label(bars, fmt="£{:,.1f}k", padding=4)

share = branch_rev.max() / branch_rev.sum()
ax.set_title(f"Windsor earns almost a third of Aldergate's revenue ({share:.0%})",
             loc="left", fontweight="bold")
ax.set_xlabel("Revenue (£k), Jan 2025–Jun 2026")
ax.set_xlim(0, branch_rev.max() * 1.18)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(left=False)
fig.text(0.02, -0.04, "Source: Aldergate till data, 4,000 orders, Jan 2025–Jun 2026.",
         fontsize=8, color="#666666")
plt.show()

**[TRACE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-trace) note:**

- T: The task is comparison, so the bars are sorted and start at zero
- R: The legend and the many colours are gone, and each value is labelled on its own bar
- C: The chart carries units, a source line, and a title that states the finding

> 🤖 **AI Lens:** Copy the code from the *draft* cell above. In the box below, replace the line in angle brackets with that code. Then send the whole box to your Associate. Compare its list with the revision above.

```text
<paste the draft cell's code here>

Critique this chart against the TRACE checklist: Task, Reduce, Accessible, Context, and Explain. What would you change?
```

**A good answer** names the meaningless colours, the redundant legend, the alphabetical order, the missing units and source, and a title that states no finding.

## Pair 2: Change over time → line chart

**The business question:** Is the shift to online real, and how fast is it moving?

This is a *change over time* task, so use a line chart. The draft gives every line its own colour and a legend. Two lines are easy to tell apart this way. With many more lines, the reader cannot match each colour to its legend entry.

> **The prompt that produced the draft:** *"Plot monthly revenue by channel over time."*

In [ ]:
# Associate's first draft
monthly = (orders.assign(month=orders["date"].dt.to_period("M").dt.to_timestamp())
                 .groupby(["month", "channel"])["order_value"].sum().unstack())

fig, ax = plt.subplots(figsize=(9, 4.5))
monthly.plot(ax=ax)
ax.set_title("Monthly Revenue by Channel")
ax.grid(True)
plt.show()

The revision drops the legend and names each line at its right-hand end with [`ax.text`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.text.html).

> **The prompt that fixed it:** *"Revise: Remove the legend. Label each line directly at its right-hand end in its own colour. In store in grey, and Online in #0072B2 to highlight it. Label the y-axis 'Revenue (£/month)' and start it at zero, with top and right spines off. Compute Online's share of revenue in the first and last month so the title can state the change."*

In [ ]:
# Human-directed revision
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(monthly.index, monthly["In store"], color="#767676", linewidth=2)
ax.plot(monthly.index, monthly["Online"], color="#0072B2", linewidth=2.5)

# Direct labels: Each line named at its end, in its own colour
ax.text(monthly.index[-1], monthly["In store"].iloc[-1], "  In store",
        color="#767676", va="center")
ax.text(monthly.index[-1], monthly["Online"].iloc[-1], "  Online",
        color="#0072B2", va="center", fontweight="bold")

first, last = monthly.iloc[0], monthly.iloc[-1]
ax.set_title(f"Online has grown from {first['Online'] / first.sum():.0%} to "
             f"{last['Online'] / last.sum():.0%} of monthly revenue",
             loc="left", fontweight="bold")
ax.set_ylabel("Revenue (£/month)")
ax.set_ylim(0, None)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**TRACE note:**

- R: The legend is gone, so the reader's eyes stay on the data
- C: The title states the finding. The y-axis starts at zero, so the reader sees each channel's size as well as its trend

The title computes its percentages from the data each time you run the cell, so they cannot go stale when the data changes. Do the same in your own titles.

## Pair 3: Relationship → scatter plot

**The business question:** Do weeks with more marketing spend see more revenue?

This is a *relationship* task, so use a scatter plot. Watch the draft's title.

> **The prompt that produced the draft:** *"Scatter plot marketing spend against revenue from the weekly `DataFrame`."*

In [ ]:
# Associate's first draft
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(weekly["marketing_spend"], weekly["revenue"])
ax.set_title("Marketing Drives Revenue")
plt.show()

The draft lacks axis labels and units, and its solid dots overlap. Its title claims that marketing *caused* the revenue, which a scatter plot cannot show. The revision uses [`np.polyfit`](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html) to measure how much revenue rises for each extra £1 of spend. The title then states that the two move together, not that one causes the other.

> **The prompt that fixed it:** *"Revise: Axis labels with units (£/week), points in #0072B2 at 75% opacity, top and right spines off. Fit the slope with numpy and put it in the title, worded as an association, explicitly not proof of cause."*

In [ ]:
# Human-directed revision
slope = np.polyfit(weekly["marketing_spend"], weekly["revenue"], 1)[0]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(weekly["marketing_spend"], weekly["revenue"], color="#0072B2", alpha=0.75)

ax.set_title(f"Higher-spend weeks see about £{slope:.2f} more revenue per £1 of spend\n"
             "(an association, which on its own does not prove that marketing caused it)",
             loc="left", fontweight="bold")
ax.set_xlabel("Marketing spend (£/week)")
ax.set_ylabel("Revenue (£/week)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**TRACE note:**

- T: The dots are partly see-through, so points that overlap stay readable
- C: The title claims only what the chart shows: How much more revenue goes with each extra £1 of spend

> 🤖 **AI Lens:** Ask your Associate the question below.

```text
Give me three reasons this correlation between marketing spend and weekly revenue might exist even if the marketing does nothing.
```

**A good answer** gives three distinct mechanisms (seasonality, a common cause such as a product launch, or spend being set from expected revenue). You could turn each one into one caveat sentence in a memo.

## Pair 4: Distribution → histogram

**The business question:** What does a *typical* Aldergate order look like, and can we say it in one sentence?

This is a *distribution* task. A histogram shows whether the mean describes a typical order well. The seaborn function [`sns.histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html) draws it.

> **The prompt that produced the draft:** *"Histogram of order_value using seaborn."*

In [ ]:
# Associate's first draft
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(orders["order_value"], ax=ax)
ax.set_title("Histogram of order_value")
plt.show()

The revision marks the median with [`ax.axvline`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axvline.html) and labels it with `ax.text`.

> **The prompt that fixed it:** *"Revise: 40 bins, #0072B2, x-axis 'Order value (£)', y-axis 'Number of orders', top and right spines off. Mark the median with a vermillion #D55E00 line and annotate it with its value in dark grey #333333. Let the title state what the median means for a typical order."*

In [ ]:
# Human-directed revision
med = orders["order_value"].median()

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(orders["order_value"], bins=40, color="#0072B2", alpha=0.85, ax=ax)

ax.axvline(med, color="#D55E00", linewidth=1.5)
ax.text(med + 1.5, ax.get_ylim()[1] * 0.92, f"median £{med:.2f}", color="#333333")

ax.set_title(f"Half of all orders are under £{med:.0f}; big baskets are rare",
             loc="left", fontweight="bold")
ax.set_xlabel("Order value (£)")
ax.set_ylabel("Number of orders")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**TRACE note:**

- T: The question is about a *typical* order. Most orders are small and a few are very large, so the mean would overstate the typical order value. The revision marks the median
- C: Both axes carry units, and the title states the finding

## Pair 5: Distribution across groups → box plot

**The business question:** Is any branch underperforming on satisfaction?

A box plot compares whole *distributions* across groups. Each box shows the median and the middle 50% of scores. The whiskers reach across most of the rest, and the dots beyond them are outliers: Scores far from the rest. The [`sns.boxplot`](https://seaborn.pydata.org/generated/seaborn.boxplot.html) function in seaborn draws it.

> **The prompt that produced the draft:** *"Box plot of satisfaction by branch with seaborn, make it colourful."*

In [ ]:
# Associate's first draft
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=orders, x="branch", y="satisfaction", hue="branch",
            palette="rainbow", legend=True, ax=ax)
ax.set_title("Satisfaction by Branch")
plt.show()

The draft has a rainbow of boxes and a legend that repeats the x-axis. The colours carry no information. The revision uses one colour and orders the boxes by median.

> **The prompt that fixed it:** *"Revise: Draw every box as an outline in Okabe-Ito blue #0072B2, no legend. Order branches by median satisfaction. Label the y-axis 'Satisfaction (0–10)', with top and right spines off. Add a title that names the outlier branch."*

In [ ]:
# Human-directed revision
rank = orders.groupby("branch")["satisfaction"].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=orders, x="branch", y="satisfaction", order=list(rank.index),
            color="#0072B2", fill=False, ax=ax)

ax.set_title("Kingston is the outlier, with lower satisfaction and wider swings",
             loc="left", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Satisfaction (0–10)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**TRACE note:**

- T: The chart compares whole distributions. Kingston's scores swing widely, so it does worse than the other branches only some of the time. A bar of averages would hide that
- R: The boxes share one colour, have no legend, and are sorted so that their order shows the ranking

## Direct labelling

Replacing legends with **direct labels** improves most of these pairs. These patterns cover nearly every case:

- **Bars:** `ax.bar_label(bars, fmt="£{:,.1f}k", padding=4)` puts a formatted value on each bar (Pair 1)
- **Line ends:** `ax.text(x_last, y_last, "  Online", color=..., va="center")` names each line at its end (Pair 2). The [design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), section "3. R: Reduce", explains when to write the label in the line's colour <!-- docs-lint: allow: color= is a matplotlib argument, code-side text -->
- **A specific point or line:** `ax.axvline(...)` plus `ax.text(...)` marks and names the thing that matters (the median in Pair 4). `ax.annotate()` adds an arrow when the label cannot sit next to its target

Direct labels count under both **R** and **A** of TRACE.

## Next steps

Every fixing prompt in this notebook has the same shape. It names the problem and the replacement, gives the colour code (such as `#0072B2`), and asks for a takeaway title. Write your [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) prompts in that shape.

> 🤖 **AI Lens:** Rehearse once before the quest. Copy the code from any revision in this notebook. In the box below, replace the line in angle brackets with that code. Then send the whole box to your Associate.

```text
<paste the revision's code here>

Run this through TRACE: Task, Reduce, Accessible, Context, Explain. Where does it still fall short?
```

**A good answer** names the letter the figure is weakest on, and what would fix it. If your Associate passes the figure on every letter, check each letter yourself.

In [Quest 06: The Chart Doctor](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/06/Quest.ipynb), you get a deck of finished charts. Each one looks professional and has at least one fault. Use TRACE to find them.